In [1]:
#reading the dataset from the bronze layer
df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("Files/sales_data_sample.csv")

#checking our dataset
print(f"Satır sayısı: {df.count()}")
print(f"Sütun sayısı: {len(df.columns)}")
df.printSchema()

StatementMeta(, 7e7d0a26-da19-4d19-a5ad-e3f18b4e07d9, 3, Finished, Available, Finished, False)

Satır sayısı: 2823
Sütun sayısı: 25
root
 |-- ORDERNUMBER: integer (nullable = true)
 |-- QUANTITYORDERED: integer (nullable = true)
 |-- PRICEEACH: double (nullable = true)
 |-- ORDERLINENUMBER: integer (nullable = true)
 |-- SALES: double (nullable = true)
 |-- ORDERDATE: string (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- QTR_ID: integer (nullable = true)
 |-- MONTH_ID: integer (nullable = true)
 |-- YEAR_ID: integer (nullable = true)
 |-- PRODUCTLINE: string (nullable = true)
 |-- MSRP: integer (nullable = true)
 |-- PRODUCTCODE: string (nullable = true)
 |-- CUSTOMERNAME: string (nullable = true)
 |-- PHONE: string (nullable = true)
 |-- ADDRESSLINE1: string (nullable = true)
 |-- ADDRESSLINE2: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- POSTALCODE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- TERRITORY: string (nullable = true)
 |-- CONTACTLASTNAME: string (nullable = true)
 |-- CON

In [3]:
display(df)

StatementMeta(, 7e7d0a26-da19-4d19-a5ad-e3f18b4e07d9, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, da56adb5-648b-48aa-9366-6102af637f4f)

In [4]:
from pyspark.sql.functions import col, to_date, upper, trim

#cleaning our dataset
df_silver = df\
    .withColumn("ORDERDATE", to_date(col("ORDERDATE"), "M/d/yyyy H:mm"))\
    .withColumn("COUNTRY", upper(trim(col("COUNTRY"))))\
    .withColumn("STATUS", upper(trim(col("STATUS"))))\
    .withColumn("PRODUCTLINE", upper(trim(col("PRODUCTLINE"))))\
    .dropDuplicates()\
    .filter(col("SALES") > 0)

#saving our dataset to silver layer as a delta table (silver_sales)
df_silver.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("silver_sales")

print(f"Silver tablosu oluşturuldu! Satır sayısı: {df_silver.count()}")

StatementMeta(, 7e7d0a26-da19-4d19-a5ad-e3f18b4e07d9, 6, Finished, Available, Finished, False)

Silver tablosu oluşturuldu! Satır sayısı: 2823


In [5]:
display(df_silver)

StatementMeta(, 7e7d0a26-da19-4d19-a5ad-e3f18b4e07d9, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8cc107b1-6d96-4f16-b6a4-9df3e1d41e28)

In [6]:
from pyspark.sql.functions import sum, avg, count, round

# Gold: sales summary based on countries
gold_country = df_silver.groupBy("COUNTRY", "YEAR_ID")\
    .agg(
        round(sum("SALES"), 2).alias("TOTAL_SALES"),
        round(avg("SALES"), 2).alias("AVG_ORDER_VALUE"),
        count("ORDERNUMBER").alias("ORDER_COUNT")
    )\
    .orderBy("YEAR_ID", "TOTAL_SALES", ascending=[True, False])

gold_country.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable("gold_country_sales")

print("Gold tablosu oluşturuldu!") #gold_countr_sales
display(gold_country)

StatementMeta(, 7e7d0a26-da19-4d19-a5ad-e3f18b4e07d9, 8, Finished, Available, Finished, False)

Gold tablosu oluşturuldu!


SynapseWidget(Synapse.DataFrame, 48aff4ea-cda6-411c-9a78-aeca92b10e02)